In [2]:
def chunk_text(text: str, max_chars: int = 800, min_chars: int = 50) -> list[str]:
    """Режет Markdown по заголовкам ##; длинные секции дробит по параграфам.
    Параметры:
        text: исходный Markdown-текст
        max_chars: максимальная длина одного чанка в символах
        min_chars: минимальная длина чанка - чанки короче выбрасываются
    Возвращает:
        Список текстовых чанков
    """
    lines = text.split("\n")
    sections, current = [], []
    for line in lines:
        if line.startswith("## ") and current:
            sections.append("\n".join(current).strip())
            current = [line]
        else:
            current.append(line)
    if current:
        sections.append("\n".join(current).strip())

    chunks = []
    for section in sections:
        if not section:
            continue
        if len(section) <= max_chars:
            chunks.append(section)
            continue
        # Длинную секцию дробим по двойным переносам (параграфам)
        buf = ""
        for paragraph in section.split("\n\n"):
            if len(buf) + len(paragraph) + 2 <= max_chars:
                buf = f"{buf}\n\n{paragraph}" if buf else paragraph
            else:
                if buf:
                    chunks.append(buf.strip())
                buf = paragraph
        if buf:
            chunks.append(buf.strip())

    return [c for c in chunks if len(c) >= min_chars]

In [3]:
from pathlib import Path

DOCS_DIR = Path("docs")

# Загрузка и чанкинг - это вы уже видели в теории
chunks = []
for path in sorted(DOCS_DIR.rglob("*")):
    if path.suffix.lower() in {".md", ".mdx"}:
        text = path.read_text(encoding="utf-8")
        for chunk in chunk_text(text):
            chunks.append({"text": chunk, "source": str(path)})

print(f"Всего чанков: {len(chunks)}")

Всего чанков: 543


In [7]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# 1. Загружаем модель эмбеддингов
embed_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# 2. Считаем эмбеддинги для всех чанков
texts = [c["text"] for c in chunks]
chunk_embeddings = embed_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype(np.float32)
print(f"Размер матрицы эмбеддингов: {chunk_embeddings.shape}")

# 3. Строим FAISS-индекс
dim = chunk_embeddings.shape[1]  # 384
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings)
print(f"В индексе {index.ntotal} векторов размерности {dim}")

# 4. Функция поиска
def vector_search(question: str, top_k: int = 3) -> list[dict]:
    query_emb = embed_model.encode(
        [question],
        normalize_embeddings=True
    ).astype(np.float32)
    scores, indices = index.search(query_emb, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "text": chunks[idx]["text"],
            "source": chunks[idx]["source"],
            "score": float(score),
        })
    return results

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Размер матрицы эмбеддингов: (543, 384)
В индексе 543 векторов размерности 384


In [8]:
question = "Как сменить папку, где хранятся модели?"
results = vector_search(question, top_k=3)

print(f"Запрос: {question}\n")
for i, r in enumerate(results, 1):
    print(f"#{i}  [{r['source']}]  score={r['score']:.3f}")
    print(r["text"][:200], "...\n")

Запрос: Как сменить папку, где хранятся модели?

#1  [docs/faq.mdx]  score=0.586
## Where are models stored?

- macOS: `~/.ollama/models`
- Linux: `/usr/share/ollama/.ollama/models`
- Windows: `C:\Users\%username%\.ollama\models`

### How do I set them to a different location?

If ...

#2  [docs/windows.mdx]  score=0.581
To change where Ollama stores the downloaded models instead of using your home directory, set the environment variable `OLLAMA_MODELS` in your user account.

1. Start the Settings (Windows 11) or Cont ...

#3  [docs/integrations/claude-desktop.mdx]  score=0.502
## Switch models

Open Ollama **Settings → Apps**. Choose your models, then select **Restart Claude**. You can use the same model for more than one option. ...

